#**Assignment 5**

###**Code Organization**.   
For easy organization and maintainability, and readability of the codebase, the different components of the transformer model were placed in their own separate files. These files are necessary for this notebook the run:
- tokenizer.py - Tokenization function
- embedding.py - Used to vectorize inputs
- layernorm.py - Layer normalization
- ffn.py - Feed forward network
- mha.py - Multi-Head Attention Mechanism
- transformer_block.py - combines layernorm, ffn and mha
- gpt_model.py - combines embedding and transformer_block
- instruction_data - Alpaca-style instruction data
- load_gpt2.py - Downloads GPT-2 weights for loading into model
- download_data.py - Download instruction dataset at "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
   
<br>
     
###**Design Choices**.  
This codebase uses the model architecture learned from chapters 2-5. Training the model from scratch did not produce any significant result, as the responses were mainly unintelligible strings of random words and punctuation. The decision was then made to use gpt-2 (124M) weights as the base model.
For finetuning, the instruction dataset used was https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json

<br>

###**Platform Used**.
Google Colab was used to create this project. Ignore the codeblock beginning with the comment "Mount current directory" - this was used to give colab access to my working directory.

<br>

###**Results/Exports**.
- The results from finetuning the model is stored at test_responses.json

<br>
<br>
<br>





In [1]:
# Mount current directory
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/A5

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/A5


In [12]:
! pip install tiktoken
! pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 766.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 103.2 MB/s eta 0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.16.0
    Uninstalling h5py-3.16.0:
      Successfully uninstalled h5py-3.16.0


In [5]:
# Import libaries

import json
import time
import torch
import torch.nn.functional as F
from functools import partial
from torch.utils.data import Dataset, DataLoader

from tokenizer import Tokenizer
from gpt_model import GPTModel
from download_data import download_instruction_data
from load_gpt2 import download_gpt2, load_weights

### Helpers

In [6]:
# Greedy Decoding - decoding strategy
def generate(model, prompt_ids, max_new_tokens, context_length, eos_id=50256):
    model.eval()
    idx = prompt_ids.clone()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = idx[:, -context_length:]
            logits = model(context)[:, -1, :]
            next_id = logits.argmax(dim=-1, keepdim=True)
            if next_id.item() == eos_id:
                break
            idx = torch.cat([idx, next_id], dim=1)
    return idx

In [7]:
# Loss calculations
def calc_loss(model, input_batch, target_batch, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    return F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        target_batch.view(-1),
        ignore_index=-100,
    )


def calc_loader_loss(model, loader, device, max_batches=None):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if max_batches and i >= max_batches:
                break
            total += calc_loss(model, x, y, device).item()
            count += 1
    model.train()
    return total / max(count, 1)


### Function to train the model

In [8]:
# Training function
def train(model, train_loader, val_loader, device, num_epochs=2,
          lr=5e-5, eval_every=25, eval_batches=5):
    model.to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)

    train_losses, val_losses = [], []
    global_step = 0
    start = time.time()

    for epoch in range(1, num_epochs + 1):
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = calc_loss(model, x, y, device)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            global_step += 1

            if global_step % eval_every == 0:
                t_loss = calc_loader_loss(model, train_loader, device, eval_batches)
                v_loss = calc_loader_loss(model, val_loader, device, eval_batches)
                train_losses.append(t_loss)
                val_losses.append(v_loss)
                elapsed = time.time() - start
                print(f"Epoch {epoch} | Step {global_step} | "
                      f"Train {t_loss:.4f} | Val {v_loss:.4f} | {elapsed:.0f}s")

    print(f"\nTraining complete — {(time.time() - start) / 60:.1f} min")
    return train_losses, val_losses

### Deliverable 1 & 2 - Instruction dataset preparation & Training pipeline with proper masking

In [9]:
# Instruction dataset preparation
def format_prompt(entry):
    text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    if entry["input"]:
        text += f"\n\n### Input:\n{entry['input']}"
    text += "\n\n### Response:\n"
    return text


def format_full(entry):
    return format_prompt(entry) + entry["output"]


class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.samples = []
        for entry in data:
            prompt_ids = tokenizer.encode(format_prompt(entry))
            full_ids = tokenizer.encode(format_full(entry))
            full_ids.append(tokenizer.eos_id)
            self.samples.append({
                "token_ids": full_ids,
                "prompt_len": len(prompt_ids),
            })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

# Training pipeline with proper masking
def collate_fn(batch, pad_id=50256, max_length=None, device="cpu"):
    max_len = max(len(s["token_ids"]) for s in batch)
    if max_length is not None:
        max_len = min(max_len, max_length)

    inputs_list, targets_list = [], []
    for sample in batch:
        ids = sample["token_ids"][:max_len]
        prompt_len = min(sample["prompt_len"], len(ids))
        padded = ids + [pad_id] * (max_len - len(ids))

        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        pad_positions = (targets == pad_id).nonzero()
        if len(pad_positions) > 1:
            targets[pad_positions[1:]] = -100

        if prompt_len > 1:
            targets[:prompt_len - 1] = -100

        inputs_list.append(inputs)
        targets_list.append(targets)

    return (
        torch.stack(inputs_list).to(device),
        torch.stack(targets_list).to(device),
    )


def load_data(path):
    with open(path) as f:
        return json.load(f)


def split_data(data, train_frac=0.85, test_frac=0.10):
    n_train = int(len(data) * train_frac)
    n_test = int(len(data) * test_frac)
    return data[:n_train], data[n_train:n_train + n_test], data[n_train + n_test:]

### Model Setup + Data Preparation

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Tokenizer
tokenizer = Tokenizer()

# Load and split data
data = load_data(download_instruction_data())
train_raw, test_raw, val_raw = split_data(data)
print(f"Train: {len(train_raw)} | Test: {len(test_raw)} | Val: {len(val_raw)}")

# Model config
cfg = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,
    "qkv_bias": True,
}

Device: cpu
Train: 935 | Test: 110 | Val: 55


### Deliverable 3 - Sample instruction-response generations

In [13]:
model = GPTModel(cfg)

# Load gpt-2 weights
settings, params = download_gpt2("124M")
load_weights(model, params)
model.eval()


total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,}")

# Datasets and loaders
my_collate = partial(collate_fn, pad_id=tokenizer.eos_id, max_length=cfg["context_length"], device=device)

train_dataset = InstructionDataset(train_raw, tokenizer)
val_dataset = InstructionDataset(val_raw, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, drop_last=True, collate_fn=my_collate)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, drop_last=False, collate_fn=my_collate)

# Sample before training
sample_entry = val_raw[0]
prompt = format_prompt(sample_entry)
prompt_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

model.to(device)

print("\n=== before training ===")
out = generate(model, prompt_ids, max_new_tokens=60,context_length=cfg["context_length"])
print(tokenizer.decode(out.squeeze(0).tolist()))

# Train the model
print("\n--- Training ---")
train_losses, val_losses = train(model, train_loader, val_loader, device, num_epochs=5, lr=1e-4, eval_every=25)

# Sample after training
print("\n=== after training ===")
out = generate(model, prompt_ids, max_new_tokens=100,
                context_length=cfg["context_length"])
print(tokenizer.decode(out.squeeze(0).tolist()))

# Generate responses for test set
print("\n--- Generating test responses ---")
results = []
for entry in test_raw[:20]:
    prompt = format_prompt(entry)
    prompt_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)
    out = generate(model, prompt_ids, max_new_tokens=128,context_length=cfg["context_length"])
    full_text = tokenizer.decode(out.squeeze(0).tolist())
    response = full_text[len(prompt):]

    results.append({
        "instruction": entry["instruction"],
        "expected": entry["output"],
        "generated": response,
    })

# Print a few examples
for i, r in enumerate(results[:5]):
    print(f"\n[{i}] {r['instruction'][:80]}")
    print(f"    Expected : {r['expected'][:80]}")
    print(f"    Generated: {r['generated'][:80]}")

# Save responses to test_responses.json
with open("test_responses.json", "w") as f:
    json.dump(results, f, indent=2)
torch.save(model.state_dict(), "model.pt")
print("\nSaved: test_responses.json, model.pt")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Parameters: 163,037,184

=== before training ===
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Convert the active sentence to passive: 'The chef cooks the meal every day.'

### Response:

Write a response that appropriately completes the request.

### Response:

Write a response that appropriately completes the request.

### Response:

Write a response that appropriately completes the request.

### Response:

Write a response that appropriately completes the request.



--- Training ---
Epoch 1 | Step 25 | Train 1.6914 | Val 1.7122 | 57s
Epoch 1 | Step 50 | Train 1.2226 | Val 1.6111 | 114s
Epoch 1 | Step 75 | Train 1.1309 | Val 1.5876 | 169s
Epoch 1 | Step 100 | Train 1.0881 | Val 1.5086 | 227s
Epoch 1 | Step 125 | Train 0.9640 | Val 1.4859 | 283s
Epoch 1 | Step 150 | Train 0.8821 | Val 1.4453 | 339s
Epoch 1 | Step 175 | Train 0.6926 | Val 1.3925 | 394s
Epoch 1 | Step 200 | Train 0.8408 | Val 1.3912 | 449s
Ep

### Deliverable 4 - Compare with a Base Model

In [14]:
print("\n--- Base vs Fine-tuned Comparison ---")

# Reload GPT-2 without fine-tuning
base_model = GPTModel(cfg).to(device)
settings, params = download_gpt2("124M")
load_weights(base_model, params)
base_model.eval()

for entry in test_raw[:5]:
    prompt = format_prompt(entry)
    prompt_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

    base_out = generate(base_model, prompt_ids, max_new_tokens=60,
                        context_length=cfg["context_length"])
    tuned_out = generate(model, prompt_ids, max_new_tokens=60,
                         context_length=cfg["context_length"])

    base_response = tokenizer.decode(base_out.squeeze(0).tolist())[len(prompt):]
    tuned_response = tokenizer.decode(tuned_out.squeeze(0).tolist())[len(prompt):]

    print(f"\nInstruction: {entry['instruction'][:80]}")
    print(f"  Expected : {entry['output'][:80]}")
    print(f"  Base     : {base_response[:80]}")
    print(f"  Tuned    : {tuned_response[:80]}")


--- Base vs Fine-tuned Comparison ---

Instruction: Rewrite the sentence using a simile.
  Expected : The car is as fast as lightning.
  Base     : 
The car is very fast.

### Error:

The car is very fast.

### Error:

The car i
  Tuned    : The car is as fast as a cheetah.

Instruction: What type of cloud is typically associated with thunderstorms?
  Expected : The type of cloud typically associated with thunderstorms is cumulonimbus.
  Base     : 
Thunderstorms are a type of thunderstorm that occurs when the cloud is thick en
  Tuned    : Thunderstorms are thunderstorms that are extremely strong and travel at high spe

Instruction: Name the author of 'Pride and Prejudice'.
  Expected : Jane Austen.
  Base     : 
Name the author of 'Pride and Prejudice'.

### Response:

Name the author of 'P
  Tuned    : The author of 'Pride and Prejudice' is Jay Gatsby.

Instruction: What is the periodic symbol for chlorine?
  Expected : The periodic symbol for chlorine is Cl.
  Base     : 
The peri